## train_test_split


In [11]:
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

In [4]:
data = load_iris()
X, y = data.data, data.target
print("The orginal size of input data:", X.shape)
print("The orginal size of o/p data:", y.shape)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.3, random_state= 42)
print("The size of i/p train set:", X_train.shape)
print("The size of i/p test set:", X_test.shape)
print("The size of o/p train set:", y_train.shape)
print("The size of o/p test set:", y_test.shape)


The orginal size of input data: (150, 4)
The orginal size of o/p data: (150,)
The size of i/p train set: (105, 4)
The size of i/p test set: (45, 4)
The size of o/p train set: (105,)
The size of o/p test set: (45,)


In [ ]:
clf = SVC(kernel= 'linear', C= 1)
clf.fit(X_train, y_train)
clf.score(X_test, y_test)
y_pred = clf.predict(X_test)

1.0

## Cross validation

In [13]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(clf, X_train,y_train, cv = 5)
scores

array([1.        , 0.95238095, 0.9047619 , 1.        , 0.95238095])

## GridSearchCV

In [14]:
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.datasets import load_iris
from sklearn.svm import SVC

In [15]:
X, y =load_iris(return_X_y= True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.3, random_state= 42)


In [16]:
svc = SVC(random_state= 42)
## define parameter grid
param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma' : ['scale', 'auto']
}
grid_search = GridSearchCV(svc, param_grid, cv= 5, scoring= 'accuracy')
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=SVC(random_state=42),
             param_grid={'C': [0.1, 1, 10], 'gamma': ['scale', 'auto'],
                         'kernel': ['linear', 'rbf']},
             scoring='accuracy')

In [17]:
# Results
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.3f}")
print(f"Test accuracy: {grid_search.score(X_test, y_test):.3f}")

Best parameters: {'C': 1, 'gamma': 'scale', 'kernel': 'linear'}
Best CV accuracy: 0.962
Test accuracy: 1.000


Linear Regression has NO hyperparameters to tune! It's a simple closed-form solution. For GridSearch, use Ridge or Lasso regression instead.

## Gridsearch on ridge

In [18]:
from sklearn.linear_model import Ridge
from sklearn.datasets import make_regression
X, y = make_regression(n_samples=1000, n_features= 5, noise= 10, random_state= 42)
X_train , X_test, y_train , y_test = train_test_split(X, y, test_size= 0.2,random_state= 42)

In [20]:
ridge = Ridge()
parameter_rige = {'alpha' : [0.01, 0.1, 1, 10, 50, 100]}
grid = GridSearchCV(ridge, parameter_rige, cv = 5, scoring= 'r2')
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=Ridge(),
             param_grid={'alpha': [0.01, 0.1, 1, 10, 50, 100]}, scoring='r2')

In [21]:
print(f"Best alpha: {grid.best_params_}")
print(f"Best CV R²: {grid.best_score_:.3f}")
print(f"Test R²: {grid.score(X_test, y_test):.3f}")

Best alpha: {'alpha': 0.1}
Best CV R²: 0.975
Test R²: 0.971


In [22]:
## similar for Lasso
from sklearn.linear_model import Lasso

lasso = Lasso(max_iter=10000)
param_grid = {'alpha': [0.001, 0.01, 0.1, 1, 10]}
grid = GridSearchCV(lasso, param_grid, cv=5, scoring='r2')
grid.fit(X_train, y_train)

print(f"Best alpha: {grid.best_params_}")
print(f"Best CV R²: {grid.best_score_:.3f}")

Best alpha: {'alpha': 0.01}
Best CV R²: 0.975


## RandomsearchCV

Why We Need uniform() and loguniform() in RandomSearchCV
The Core Problem: Regular Parameters vs ML Hyperparameters
Regular numbers (like 1, 2, 3, 4) are linear. ML hyperparameters often work on logarithmic scales. Using regular uniform sampling would waste most of your searches!

In [23]:
from sklearn.model_selection import  RandomizedSearchCV
from scipy.stats import uniform, loguniform
import numpy as np

# Your data
X, y = make_regression(n_samples=1000, n_features=5, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define model
ridge = Ridge(random_state=42)

# Define parameter distribution (not grid!)
param_dist = {
    'alpha': loguniform(1e-3, 1e3),  # Continuous distribution from 0.001 to 1000
    'solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sag', 'saga'],
    'tol': uniform(1e-5, 1e-3)       # Continuous from 0.00001 to 0.00101
}

# Random Search
random_search = RandomizedSearchCV(
    ridge, 
    param_dist, 
    n_iter=20,           # Try 20 random combinations
    cv=5,                # 5-fold cross validation
    scoring='r2',
    random_state=42,
    n_jobs=-1            # Use all CPU cores
)

# Fit
random_search.fit(X_train, y_train)

# Results
print("="*50)
print("RANDOMIZED SEARCH CV RESULTS")
print("="*50)
print(f"Best parameters: {random_search.best_params_}")
print(f"Best CV R² score: {random_search.best_score_:.4f}")
print(f"Test R² score: {random_search.score(X_test, y_test):.4f}")

RANDOMIZED SEARCH CV RESULTS
Best parameters: {'alpha': np.float64(0.4374364439939078), 'solver': 'lsqr', 'tol': np.float64(0.0005051769101112702)}
Best CV R² score: 0.9751
Test R² score: 0.9710
